In [1]:
# Cell 1 — Read nested JSON and explode
from pyspark.sql import functions as F
from pyspark.sql import types as T

raw = spark.read \
    .option("multiline", "true") \
    .json("Files/raw/products/products_catalog.json")

# Products are inside a "data" array — explode it into rows
products_df = raw.select(F.explode(F.col("data")).alias("p")).select("p.*")

print("Products:", products_df.count())
products_df.printSchema()


StatementMeta(, b6657dfb-5e03-44e8-a1bc-12947398213a, 3, Finished, Available, Finished, False)

Products: 200
root
 |-- base_price: double (nullable = true)
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- review_count: long (nullable = true)
 |-- selling_price: double (nullable = true)
 |-- stock_quantity: long (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- weight_kg: double (nullable = true)



In [4]:
# Clean and transform product data
cleaned = (
    products_df
    .withColumn("base_price", F.col("base_price").cast(T.DecimalType(12, 2)))
    .withColumn("selling_price", F.col("selling_price").cast(T.DecimalType(12, 2)))
    .withColumn("created_at", F.to_timestamp(F.col("created_at")))
    .withColumn(
        "price_anomaly",
        F.col("selling_price") > F.col("base_price")
    )
    .withColumn("silver_created_at", F.current_timestamp())
)

# Silver Lakehouse Delta path
TARGET_PATH = (
    "abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/"
    "silver_lakehouse.Lakehouse/Tables/silver_products"
)

# Write as Delta
(
    cleaned.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(TARGET_PATH)
)

# Validate
silver_products = (
    spark.read
    .format("delta")
    .load(TARGET_PATH)
)

row_count = silver_products.count()

print("=" * 60)
print("✅ Silver Products table created successfully!")
print(f"📂 Path      : {TARGET_PATH}")
print(f"📊 Total Rows: {row_count}")
print("=" * 60)

silver_products.show(10, truncate=False)

StatementMeta(, b6657dfb-5e03-44e8-a1bc-12947398213a, 6, Finished, Available, Finished, False)

✅ Silver Products table created successfully!
📂 Path      : abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/silver_lakehouse.Lakehouse/Tables/silver_products
📊 Total Rows: 200
+----------+---------+--------------+-------------------+---------+----------+--------------------------------+------+------------+-------------+--------------+-----------+---------+-------------+--------------------------+
|base_price|brand    |category      |created_at         |is_active|product_id|product_name                    |rating|review_count|selling_price|stock_quantity|supplier_id|weight_kg|price_anomaly|silver_created_at         |
+----------+---------+--------------+-------------------+---------+----------+--------------------------------+------+------------+-------------+--------------+-----------+---------+-------------+--------------------------+
|19684.23  |Apple    |Books         |2023-01-31 00:00:00|true     |PROD1000  |Noise Books Model-544           |1.4   |88          |191